# NB08 Proof of work — Chapter 8

A chapter notebook for *Blockchain for Lawyers: A Reading Through Technology and Economics*. The code is the same as in the text.

- All code here is a **toy implementation** meant to make a principle visible. Do not use it with real assets, real private keys or real transactions.
- No external API is called, and everything runs without a network. Cells that need the bundled snapshot data carry that data inside the cell.
- Standard library only (run and checked on Python 3.13). There is no install cell.
- The outputs are the editor's own run, stored as produced. Running the cells again should give the same values (random seeds are fixed).

In [ ]:
# NB08-1: a hand miner — how many times must the nonce change before a hash with k leading zero bits turns up? (for study; toy difficulty)
import hashlib

def mine(header_prefix: bytes, bits: int):
    """Raise the nonce from 0 and find a nonce for which the first `bits` bits of SHA256(SHA256(prefix||nonce)) are 0. Returns (nonce, number of attempts, hash)."""
    target = 1 << (256 - bits)                 # "first `bits` bits are 0" ⇔ hash < 2^(256-bits)
    nonce = 0
    while True:
        h = hashlib.sha256(hashlib.sha256(header_prefix + nonce.to_bytes(4, "little")).digest()).digest()
        if int.from_bytes(h, "big") < target:
            return nonce, nonce + 1, h
        nonce += 1

prefix = b"block#1|prev=00..|merkle=ab..|time=1231006505|"
print(f"{'leading 0 bits k':>16} {'nonce found':>11} {'attempts':>10} {'expected 2^k':>12} {'first 8 hex':>11}")
for bits in [4, 8, 12, 16, 20]:
    nonce, tries, h = mine(prefix, bits)
    print(f"{bits:>16} {nonce:>11,} {tries:>10,} {2**bits:>12,} {h.hex()[:8]:>11}")

leading 0 bits k nonce found   attempts expected 2^k first 8 hex
               4           3          4           16    04468f48
               8         329        330          256    004737bc
              12       6,665      6,666        4,096    000c395c
              16      24,541     24,542       65,536    00002bbf
              20     978,082    978,083    1,048,576    00000fc3


In [ ]:
# NB08-2: mine 200 blocks at the same difficulty (k=12) — the distribution of attempts (geometric distribution)
import math
bits = 12
tries_list = []
for i in range(200):
    _, tries, _ = mine(f"block#{i}|".encode() + prefix, bits)
    tries_list.append(tries)
tries_list.sort()
n = len(tries_list)
mean = sum(tries_list) / n
print(f"k={bits}: expected attempts 2^k = {2**bits:,}")
print(f"200 runs: mean {mean:,.0f}, median {tries_list[n//2]:,}, min {tries_list[0]:,}, max {tries_list[-1]:,}")
print(f"geometric distribution, theory: mean {2**bits:,}, median ≈ ln2·2^k = {math.log(2)*2**bits:,.0f}, standard deviation ≈ 2^k = {2**bits:,}")
quick = sum(1 for t in tries_list if t <= 2**bits * 0.1)
slow = sum(1 for t in tries_list if t >= 2**bits * 3)
print(f"blocks done within 1/10 of the expected value: {quick} ({quick/2:.0f}%) | blocks that took 3 times the expected value or more: {slow} ({slow/2:.0f}%)")

k=12: expected attempts 2^k = 4,096
200 runs: mean 4,061, median 3,058, min 27, max 25,448
geometric distribution, theory: mean 4,096, median ≈ ln2·2^k = 2,839, standard deviation ≈ 2^k = 4,096
blocks done within 1/10 of the expected value: 23 (12%) | blocks that took 3 times the expected value or more: 9 (4%)


In [ ]:
# NB08-3: nBits → target → difficulty → hash-rate estimate. Genesis (0x1d00ffff) and the chain tip on 2026-10-25 (0x17020093)
import math
def nbits_to_target(nbits: int) -> int:
    exponent, coefficient = nbits >> 24, nbits & 0x007FFFFF
    return coefficient * 256 ** (exponent - 3)

T_MAX = nbits_to_target(0x1d00ffff)              # the easiest target (genesis nBits; equal to powLimit reduced to compact form)
print("T_max = 0x%064x" % T_MAX)
print("expected hashes per block at difficulty 1, 2^256/T_max = %.6f × 2^32" % (2**256 / T_MAX / 2**32))

for name, nbits in [("Genesis (2009-01-03)", 0x1d00ffff), ("Height 972,958 (as of 2026-10-25)", 0x17020093)]:
    T = nbits_to_target(nbits)
    D = T_MAX / T
    p = (T + 1) / 2**256
    print(f"\n{name}: nBits = 0x{nbits:08x}")
    print("  target T = 0x%064x" % T)
    print("  probability that one hash succeeds p = %.3e  (leading 0 bits ≈ %.1f)" % (p, 256 - math.log2(T + 1)))
    print("  difficulty D = T_max/T = %.1f" % D)
    print("  expected hashes per block ≈ D·2^32 = %.3e" % (D * 2**32))
    H = D * 2**32 / 600
    print("  hash-rate estimate assuming 600 s, D·2^32/600 = %.4e H/s" % H + (" = %.0f EH/s" % (H / 1e18) if H >= 1e15 else " = %.1f MH/s" % (H / 1e6)))

T_max = 0x00000000ffff0000000000000000000000000000000000000000000000000000
expected hashes per block at difficulty 1, 2^256/T_max = 1.000015 × 2^32

Genesis (2009-01-03): nBits = 0x1d00ffff
  target T = 0x00000000ffff0000000000000000000000000000000000000000000000000000
  probability that one hash succeeds p = 2.328e-10  (leading 0 bits ≈ 32.0)
  difficulty D = T_max/T = 1.0
  expected hashes per block ≈ D·2^32 = 4.295e+09
  hash-rate estimate assuming 600 s, D·2^32/600 = 7.1583e+06 H/s = 7.2 MH/s

Height 972,958 (as of 2026-10-25): nBits = 0x17020093
  target T = 0x0000000000000000000200930000000000000000000000000000000000000000
  probability that one hash succeeds p = 1.656e-24  (leading 0 bits ≈ 79.0)
  difficulty D = T_max/T = 140577680051919.6
  expected hashes per block ≈ D·2^32 = 6.038e+23
  hash-rate estimate assuming 600 s, D·2^32/600 = 1.0063e+21 H/s = 1006 EH/s


In [ ]:
# NB08-4: simulating the adjustment rule — if the hash rate suddenly becomes 16 times larger, how many adjustments does difficulty need to catch up? (the Core rule as it is)
TIMESPAN, SPACING, INTERVAL = 1_209_600, 600, 2016      # the values in chainparams.cpp and params.h
def retarget(T_old: int, actual_timespan: int) -> int:
    """pow.cpp CalculateNextWorkRequired: clamp the time taken to [timespan/4, timespan*4], adjust in proportion, cap at powLimit."""
    actual = max(TIMESPAN // 4, min(TIMESPAN * 4, actual_timespan))
    T_new = T_old * actual // TIMESPAN
    return min(T_new, T_MAX)

T = T_MAX // 1000                      # starting from some difficulty (1000)
H1 = 2**256 / (T + 1) / 600            # take as 1 the hash rate (hashes/second) that gives exactly 600 seconds per block at that difficulty
hashrate = 1.0
print(f"{'round':>6} {'difficulty':>10} {'mean block interval (s)':>23} {'next difficulty':>15}")
for period in range(1, 7):
    if period == 2:
        hashrate = 16.0                # from the round-2 period on, the hash rate jumps 16-fold
    D = T_MAX / T
    expected_hashes_per_block = 2**256 / (T + 1)
    seconds_per_block = expected_hashes_per_block / (hashrate * H1)
    actual_timespan = int(seconds_per_block * INTERVAL)
    T_next = retarget(T, actual_timespan)
    print(f"{period:>6} {D:>10,.0f} {seconds_per_block:>23,.0f} {T_MAX / T_next:>15,.0f}")
    T = T_next

 round difficulty mean block interval (s) next difficulty
     1      1,000                     600           1,000
     2      1,000                      38           4,000
     3      4,000                     150          16,000
     4     16,000                     600          16,000
     5     16,000                     600          16,000
     6     16,000                     600          16,000
